# HKI + MCP: Hermetic Tool Routers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h3nok/HKI/blob/main/notebooks/06_mcp_server.ipynb)

Model Context Protocol (MCP) is the dominant standard for exposing tools to LLM agents. Without HKI, any MCP server is a cross-tenant attack surface: a `payments` agent can freely call `hr.get_salary_band()` or `legal.get_case_notes()` just because the server exposes those tools.

This notebook shows:

1. **The problem** — a bare MCP server leaks across knowledge domains
2. **Section 1** — HKI envelope validation at the MCP router layer
3. **Section 2** — `HkiToolGuard` as the tool-level enforcement primitive
4. **Section 3** — Domain-tagged tool registry (tools declare their domain)
5. **Section 4** — Multi-domain agent: approved cross-domain publish pattern
6. **Section 5** — `hki-conformance` probe against an MCP server endpoint
7. **Section 6** — Production wiring: FastMCP + HKI middleware

In [ ]:
%pip install hki-runtime hki-adk -q
# MCP SDK (official Python SDK)
%pip install mcp -q

In [ ]:
import json, time, uuid
import hki_runtime
import hki_adk

# ── Envelope factory ───────────────────────────────────────────────────────
def make_envelope(domain: str, *, purpose: str = "tool-call", ttl: int = 300) -> dict:
    now = int(time.time())
    return {
        "hki_version": "1.0",
        "envelope_id": str(uuid.uuid4()),
        "org_id": "org_demo",
        "subject_id": "user_alice",
        "active_domain": domain,
        "authorized_domains": [domain],
        "purpose": purpose,
        "risk_tier": "read-only",
        "policy_pack_id": f"pp_{domain}_v1",
        "issuer": "urn:hki:demo-gateway",
        "signature": f"demo-sig-{domain}",
        "issued_at": now,
        "expires_at": now + ttl,
    }

print("Envelope factory ready")

---
## The Problem — MCP Without HKI

A bare MCP server exposes all tools to all callers. The LLM decides which tool to invoke — and it can decide wrong.

In [ ]:
# ── Vulnerable MCP-style dispatcher (no isolation) ─────────────────────────
TOOL_REGISTRY_BARE = {
    "payments.process_refund": lambda amount, order_id: f"refund_{order_id}_{amount}",
    "hr.get_salary_band": lambda employee_id: f"salary_band_senior_for_{employee_id}",
    "legal.get_case_notes": lambda case_id: f"confidential_notes_for_{case_id}",
}

def bare_dispatch(tool_name: str, **kwargs):
    """No authentication, no scope check — just call whatever is asked."""
    fn = TOOL_REGISTRY_BARE.get(tool_name)
    if fn is None:
        return {"error": f"tool {tool_name!r} not found"}
    return {"result": fn(**kwargs)}

# A payments agent calls hr and legal tools — and it works.
print("payments agent calls hr tool  :", bare_dispatch("hr.get_salary_band", employee_id="emp_101"))
print("payments agent calls legal tool:", bare_dispatch("legal.get_case_notes", case_id="case_999"))
print()
print("This is HKI-T05 (MCP scope escape). Any domain can call any tool.")

---
## Section 1 — Envelope Validation at the MCP Router

The fix starts at the router layer: every tool call must arrive with a valid HKI envelope, and the envelope's `active_domain` determines which tools are reachable.

In [ ]:
# ── HKI-aware MCP router ───────────────────────────────────────────────────
class HkiMcpRouter:
    """Minimal MCP router that enforces HKI before dispatching."""

    def __init__(self):
        # domain → {tool_name → callable}
        self._tools: dict[str, dict[str, callable]] = {}

    def register(self, domain: str, tool_name: str, fn: callable):
        self._tools.setdefault(domain, {})[tool_name] = fn

    def call(self, envelope_dict: dict, tool_name: str, **kwargs) -> dict:
        # 1. Validate envelope
        result = hki_runtime.validate_envelope(envelope_dict, require_signature=True)
        if not result.ok:
            codes = [i.code for i in result.issues]
            return {"error": "envelope_invalid", "codes": codes}

        envelope = result.envelope

        # 2. Reject scope-override args
        scope_err = hki_runtime.reject_conflicting_scope_argument(envelope, kwargs)
        if scope_err:
            return {"error": "scope_override", "detail": scope_err}

        # 3. Look up tool by active_domain
        domain_tools = self._tools.get(envelope.active_domain, {})
        fn = domain_tools.get(tool_name)
        if fn is None:
            return {
                "error": "tool_not_found",
                "detail": f"{tool_name!r} not available in domain {envelope.active_domain!r}",
            }

        # 4. Execute
        return {"result": fn(**kwargs), "domain": envelope.active_domain}


router = HkiMcpRouter()
router.register("payments", "payments.process_refund",
                lambda amount, order_id: f"refund_{order_id}_{amount}")
router.register("hr",       "hr.get_salary_band",
                lambda employee_id: f"salary_band_senior_for_{employee_id}")
router.register("legal",    "legal.get_case_notes",
                lambda case_id: f"confidential_notes_for_{case_id}")

# ── Payments agent with payments envelope ─────────────────────────────────
payments_env = make_envelope("payments")
r1 = router.call(payments_env, "payments.process_refund", amount=99.99, order_id="ord_555")
print("Allowed — payments tool:", r1)

# ── Same agent tries to reach hr tool ─────────────────────────────────────
r2 = router.call(payments_env, "hr.get_salary_band", employee_id="emp_101")
print("Blocked — hr tool from payments domain:", r2)

---
## Section 2 — HkiToolGuard: Tool-Level Enforcement

`HkiToolGuard` from `hki_adk` wraps any callable and enforces HKI from the ADK `ToolContext`. It's the same primitive used in the production orchestrator service.

In [ ]:
# ── Simulate ADK ToolContext ───────────────────────────────────────────────
class FakeToolContext:
    def __init__(self, envelope_dict: dict):
        self.state = {"hki_envelope": envelope_dict}

# ── Raw tool — no isolation ────────────────────────────────────────────────
def raw_hr_lookup(employee_id: str) -> str:
    return f"salary_band_senior_for_{employee_id}"

# ── Wrap with HkiToolGuard ─────────────────────────────────────────────────
# domain="hr" means only envelopes with active_domain="hr" may call this tool
guarded_hr_lookup = hki_adk.HkiToolGuard(
    raw_hr_lookup,
    domain="hr",
)

# Authorized call
hr_env = make_envelope("hr")
hr_ctx = FakeToolContext(hr_env)
try:
    result = guarded_hr_lookup(employee_id="emp_101", tool_context=hr_ctx)
    print("Allowed (hr domain):", result)
except hki_adk.HkiAdkDenied as e:
    print("Denied:", e)

# Cross-domain attack
payments_ctx = FakeToolContext(make_envelope("payments"))
try:
    result = guarded_hr_lookup(employee_id="emp_101", tool_context=payments_ctx)
    print("LEAK:", result)
except hki_adk.HkiAdkDenied as e:
    print("Blocked (payments→hr):", e)

---
## Section 3 — Domain-Tagged Tool Registry

In production, each tool declares the domain it belongs to. The MCP router uses `evaluate_gateway_target` to check whether the active domain may call a given target.

In [ ]:
# ── Tool registry with domain metadata ────────────────────────────────────
TOOL_REGISTRY = [
    {
        "name": "payments.process_refund",
        "domain": "payments",
        "published_domains": [],   # not published externally
        "fn": lambda amount, order_id: f"refund_{order_id}_{amount}",
    },
    {
        "name": "hr.get_salary_band",
        "domain": "hr",
        "published_domains": [],
        "fn": lambda employee_id: f"salary_band_senior",
    },
    {
        "name": "retrieval.search",
        "domain": "payments",
        "published_domains": ["finance", "audit"],  # allowed cross-domain readers
        "fn": lambda query: f"results_for_{query}",
    },
]

def gateway_dispatch(envelope_dict: dict, tool_name: str, **kwargs) -> dict:
    result = hki_runtime.validate_envelope(envelope_dict, require_signature=True)
    if not result.ok:
        return {"error": "envelope_invalid", "codes": [i.code for i in result.issues]}
    envelope = result.envelope

    tool_meta = next((t for t in TOOL_REGISTRY if t["name"] == tool_name), None)
    if tool_meta is None:
        return {"error": "not_found", "tool": tool_name}

    target = hki_runtime.HkiGatewayTarget(
        type="tool",
        id=tool_name,
        domain=tool_meta["domain"],
        published_domains=tuple(tool_meta["published_domains"]),
    )
    decision = hki_runtime.evaluate_gateway_target(envelope, target)
    if not decision.allowed:
        return {"error": "denied", "reason": decision.reason}

    return {"result": tool_meta["fn"](**kwargs), "via": decision.reason}

payments_env = make_envelope("payments")
finance_env  = make_envelope("finance")

print("payments → payments.process_refund:",
      gateway_dispatch(payments_env, "payments.process_refund", amount=50, order_id="o1"))

print("payments → hr.get_salary_band    :",
      gateway_dispatch(payments_env, "hr.get_salary_band", employee_id="e1"))

# finance is in published_domains of retrieval.search — this should be allowed
print("finance  → retrieval.search      :",
      gateway_dispatch(finance_env, "retrieval.search", query="refund policy"))

# hr is NOT in published_domains
hr_env = make_envelope("hr")
print("hr       → retrieval.search      :",
      gateway_dispatch(hr_env, "retrieval.search", query="refund policy"))

---
## Section 4 — Multi-Domain Agent: Approved Cross-Domain Publish

Some agents legitimately write across domains — an audit pipeline that reads from `payments` and writes a summary to `audit`. HKI models this through `authorized_domains` (all domains the agent may read) and per-tool `published_domains` (domains that may receive the output). The agent never gets unscoped access.

In [ ]:
# ── Multi-domain envelope: audit agent may read payments + audit ──────────
now = int(time.time())
audit_envelope = {
    "hki_version": "1.0",
    "envelope_id": str(uuid.uuid4()),
    "org_id": "org_demo",
    "subject_id": "agent_audit_pipeline",
    "active_domain": "payments",           # reading from payments now
    "authorized_domains": ["payments", "audit"],  # may switch to audit later
    "purpose": "review",
    "risk_tier": "read-only",
    "policy_pack_id": "pp_audit_v1",
    "issuer": "urn:hki:audit-gateway",
    "signature": "demo-audit-sig",
    "issued_at": now,
    "expires_at": now + 600,
}

# ── Step 1: read from payments ─────────────────────────────────────────────
r1 = gateway_dispatch(audit_envelope, "payments.process_refund",
                      amount=0, order_id="read_only")
print("Step 1 — read payments tool:", r1)

# ── Step 2: switch active_domain to audit to write summary ────────────────
# In production the gateway mints a new envelope with active_domain="audit".
# Here we simulate by creating the second envelope directly.
audit_write_envelope = {
    **audit_envelope,
    "envelope_id": str(uuid.uuid4()),
    "active_domain": "audit",
    "purpose": "publish",
    "risk_tier": "write",
    "signature": "demo-audit-write-sig",
    "issued_at": now,
    "expires_at": now + 600,
}

# Confirm audit write envelope is valid
v = hki_runtime.validate_envelope(audit_write_envelope, require_signature=True)
print("\nAudit write envelope valid:", v.ok)
print("active_domain:", v.envelope.active_domain)
print("authorized_domains:", v.envelope.authorized_domains)

# ── Step 3: scope-override attempt is caught ───────────────────────────────
# An attacker injects domain into the tool kwargs to escape scope.
attack_kwargs = {"active_domain": "hr", "query": "salary data"}
scope_err = hki_runtime.reject_conflicting_scope_argument(v.envelope, attack_kwargs)
print("\nScope override blocked:", scope_err)

---
## Section 5 — `hki-conformance` Probe

The `@hki/conformance` package ships a CLI probe that runs all envelope and visibility conformance cases against any adapter. The Python runtime exposes the same contract, so we can run the conformance suite directly.

In [ ]:
# ── Run a subset of the conformance cases in Python ───────────────────────
# These mirror the cases in packages/hki-conformance/src/cases.ts.

CONFORMANCE_CASES = [
    {
        "id": "ENV-001",
        "description": "valid envelope is accepted",
        "input": make_envelope("payments"),
        "expect_ok": True,
    },
    {
        "id": "ENV-002",
        "description": "expired envelope is rejected",
        "input": {**make_envelope("payments"), "expires_at": int(time.time()) - 1},
        "expect_ok": False,
    },
    {
        "id": "ENV-003",
        "description": "global active_domain is rejected",
        "input": {**make_envelope("payments"), "active_domain": "global",
                  "authorized_domains": ["global"]},
        "expect_ok": False,
    },
    {
        "id": "ENV-004",
        "description": "active_domain not in authorized_domains is rejected",
        "input": {**make_envelope("payments"), "authorized_domains": ["hr"]},
        "expect_ok": False,
    },
    {
        "id": "ENV-005",
        "description": "missing signature is rejected when required",
        "input": {**make_envelope("payments"), "signature": ""},
        "expect_ok": False,
    },
    {
        "id": "VIS-001",
        "description": "artifact in active_domain is visible",
        "input": "visibility",
        "envelope_domain": "payments",
        "artifact_domain": "payments",
        "expect_ok": True,
    },
    {
        "id": "VIS-002",
        "description": "artifact in another domain is not visible",
        "input": "visibility",
        "envelope_domain": "payments",
        "artifact_domain": "hr",
        "expect_ok": False,
    },
]

passed = 0
failed = 0

for case in CONFORMANCE_CASES:
    if case["input"] == "visibility":
        env_dict = make_envelope(case["envelope_domain"])
        v = hki_runtime.validate_envelope(env_dict, require_signature=True)
        envelope = v.envelope
        label = hki_runtime.HkiArtifactLabel(
            org_id="org_demo",
            domain=case["artifact_domain"],
            artifact_type="document",
            artifact_id="doc_1",
        )
        issue = hki_runtime.assert_artifact_visible(envelope, label)
        ok = issue is None
    else:
        v = hki_runtime.validate_envelope(case["input"], require_signature=True)
        ok = v.ok

    actual_pass = ok == case["expect_ok"]
    status = "PASS" if actual_pass else "FAIL"
    if actual_pass:
        passed += 1
    else:
        failed += 1

    print(f"{status}  [{case['id']}] {case['description']}")

print(f"\n{passed} passed, {failed} failed")

---
## Section 6 — Production Wiring: FastMCP + HKI Middleware

In production the MCP server is a FastAPI app with `HkiMiddleware` injected at the ASGI layer. Every tool call goes through envelope extraction → validation → domain routing. The middleware lives in `packages/hki-runtime-py/hki_runtime/fastapi.py`.

In [ ]:
from hki_runtime.fastapi import HkiMiddleware
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
import json

# ── Build the MCP-style FastAPI server ────────────────────────────────────
app = FastAPI(title="HKI MCP Server")
app.add_middleware(HkiMiddleware)  # enforces envelope on every request


@app.post("/mcp/tools/call")
async def mcp_tool_call(request: Request):
    body = await request.json()
    tool_name = body.get("tool")
    kwargs = body.get("args", {})

    # Envelope is already validated by HkiMiddleware and attached to request state
    envelope: hki_runtime.HkiEnvelope = request.state.hki_envelope

    # Simple domain-aware routing
    domain_tools = {
        "payments": {
            "process_refund": lambda amount, order_id: f"refund {order_id} €{amount}",
        },
        "hr": {
            "get_headcount": lambda dept: f"{dept} has 42 employees",
        },
    }

    domain = envelope.active_domain
    fn = (domain_tools.get(domain) or {}).get(tool_name)
    if fn is None:
        from fastapi.responses import JSONResponse
        return JSONResponse(
            status_code=403,
            content={"error": "tool_not_found", "domain": domain, "tool": tool_name},
        )

    return {"result": fn(**kwargs), "domain": domain, "envelope_id": envelope.envelope_id}


client = TestClient(app, raise_server_exceptions=False)

def call_tool(envelope_dict: dict, tool: str, args: dict) -> dict:
    resp = client.post(
        "/mcp/tools/call",
        json={"tool": tool, "args": args},
        headers={"x-hki-envelope": json.dumps(envelope_dict)},
    )
    return {"status": resp.status_code, **resp.json()}

payments_env = make_envelope("payments")
hr_env = make_envelope("hr")

print("payments agent → payments tool:", call_tool(payments_env, "process_refund", {"amount": 29.99, "order_id": "ord_7"}))
print("payments agent → hr tool      :", call_tool(payments_env, "get_headcount", {"dept": "engineering"}))
print("hr agent → hr tool            :", call_tool(hr_env, "get_headcount", {"dept": "engineering"}))
print("no envelope                   :", client.post("/mcp/tools/call", json={"tool": "get_headcount", "args": {}}).status_code)

---
## Summary

| Layer | Primitive | What it stops |
|-------|-----------|---------------|
| ASGI middleware | `HkiMiddleware` | Requests with no/invalid envelope |
| MCP router | `validate_envelope` + domain routing | Cross-domain tool access |
| Tool level | `HkiToolGuard` | Per-tool domain enforcement |
| Gateway | `evaluate_gateway_target` | Unauthorized published-domain reads |
| Args | `reject_conflicting_scope_argument` | Body-injection scope override |
| Conformance | `hki-conformance` suite | Regressions in any adapter |

The MCP protocol itself provides no isolation — HKI adds the domain boundary that makes MCP servers safe to expose to multi-tenant agentic workloads.